# March Madness 2024


### Simple Usage of previous March Madness 2022 Xgboost 
- Original solution : https://www.kaggle.com/code/cv13j0/ncaa-gradient-boosted-trees-xgboost#Creating-the-Test-Dataset by [@cv13j0](https://www.kaggle.com/cv13j0)

- Preprocessing code : https://www.kaggle.com/code/benjenkins96/deep-learning-techniques-to-predict-march-madness by [@benjenkins96](https://www.kaggle.com/benjenkins96)

- How to apply previous solutions to March Madness 2024 : https://www.kaggle.com/code/samdaltonjr/preliminary-preds-into-bracket by [@samdaltonjr](https://www.kaggle.com/samdaltonjr)

Thanks for both of u for making these amazing notebooks.

# Import Libraries

In [ ]:
import os
import re
import sklearn
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.metrics import *
from sklearn.linear_model import *
from sklearn.model_selection import *

pd.set_option('display.max_columns', None)
DATA_PATH = '/kaggle/input/march-machine-learning-mania-2024/'

for filename in sorted(os.listdir(DATA_PATH)):
    print(filename)

# Train data - df

- Preprocessing code : https://www.kaggle.com/code/benjenkins96/deep-learning-techniques-to-predict-march-madness by [@benjenkins96](https://www.kaggle.com/benjenkins96)


In [ ]:
df_seeds = pd.concat([
    pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv"),
    pd.read_csv(DATA_PATH + "WNCAATourneySeeds.csv"),
], ignore_index=True)
    
df_seeds.head()

In [ ]:
df_season_results = pd.concat([
    pd.read_csv(DATA_PATH + "MRegularSeasonCompactResults.csv"),
    pd.read_csv(DATA_PATH + "WRegularSeasonCompactResults.csv"),
], ignore_index=True)

df_season_results.drop(['NumOT', 'WLoc'], axis=1, inplace=True)
df_season_results

### 

In [ ]:
# scoregap between W & L
df_season_results['ScoreGap'] = df_season_results['WScore'] - df_season_results['LScore']
df_season_results.head()


In [ ]:
# each team's # of wins 
num_win = df_season_results.groupby(['Season', 'WTeamID']).count()
num_win = num_win.reset_index()[['Season', 'WTeamID', 'DayNum']].rename(columns={"DayNum": "NumWins", "WTeamID": "TeamID"})

In [ ]:
num_win

In [ ]:
# each team's # of loss
num_loss = df_season_results.groupby(['Season', 'LTeamID']).count()
num_loss = num_loss.reset_index()[['Season', 'LTeamID', 'DayNum']].rename(columns={"DayNum": "NumLosses", "LTeamID": "TeamID"})

In [ ]:
num_loss

In [ ]:
# how much points they scored more in average
gap_win = df_season_results.groupby(['Season', 'WTeamID']).mean().reset_index()
gap_win = gap_win[['Season', 'WTeamID', 'ScoreGap']].rename(columns={"ScoreGap": "GapWins", "WTeamID": "TeamID"})

In [ ]:
gap_win

In [ ]:
# how much points they scored less in average
gap_loss = df_season_results.groupby(['Season', 'LTeamID']).mean().reset_index()
gap_loss = gap_loss[['Season', 'LTeamID', 'ScoreGap']].rename(columns={"ScoreGap": "GapLosses", "LTeamID": "TeamID"})

In [ ]:
gap_loss

In [ ]:
df_features_season_w = df_season_results.groupby(['Season', 'WTeamID']).count().reset_index()[['Season', 'WTeamID']].rename(columns={"WTeamID": "TeamID"})
df_features_season_l = df_season_results.groupby(['Season', 'LTeamID']).count().reset_index()[['Season', 'LTeamID']].rename(columns={"LTeamID": "TeamID"})

In [ ]:
df_features_season_w

In [ ]:
df_features_season_l

In [ ]:
df_features_season = pd.concat([df_features_season_w, df_features_season_l], axis=0).drop_duplicates().sort_values(['Season', 'TeamID']).reset_index(drop=True)

In [ ]:
df_features_season

In [ ]:
df_features_season = df_features_season.merge(num_win, on=['Season', 'TeamID'], how='left')
df_features_season = df_features_season.merge(num_loss, on=['Season', 'TeamID'], how='left')
df_features_season = df_features_season.merge(gap_win, on=['Season', 'TeamID'], how='left')
df_features_season = df_features_season.merge(gap_loss, on=['Season', 'TeamID'], how='left')

In [ ]:
df_features_season.fillna(0, inplace=True)  


In [ ]:
df_features_season['WinRatio'] = df_features_season['NumWins'] / (df_features_season['NumWins'] + df_features_season['NumLosses'])
df_features_season['GapAvg'] = (
    (df_features_season['NumWins'] * df_features_season['GapWins'] - 
    df_features_season['NumLosses'] * df_features_season['GapLosses'])
    / (df_features_season['NumWins'] + df_features_season['NumLosses'])
)

In [ ]:
df_features_season.drop(['NumWins', 'NumLosses', 'GapWins', 'GapLosses'], axis=1, inplace=True)

In [ ]:
df_features_season

In [ ]:
df_tourney_results = pd.concat([
    pd.read_csv(DATA_PATH + "WNCAATourneyCompactResults.csv"),
    pd.read_csv(DATA_PATH + "MNCAATourneyCompactResults.csv"),
], ignore_index=True)
df_tourney_results.drop(['NumOT', 'WLoc'], axis=1, inplace=True)

In [ ]:
df = df_tourney_results.copy()
df = df[df['Season'] >= 2016].reset_index(drop=True)

df.head()

In [ ]:
df = pd.merge(
    df, 
    df_seeds,   # TourneySeeds (M+W) (season / seed / teamid)
    how='left', 
    left_on=['Season', 'WTeamID'], 
    right_on=['Season', 'TeamID']
).drop('TeamID', axis=1).rename(columns={'Seed': 'SeedW'})

In [ ]:
df = pd.merge(
    df, 
    df_seeds, 
    how='left', 
    left_on=['Season', 'LTeamID'], 
    right_on=['Season', 'TeamID']
).drop('TeamID', axis=1).rename(columns={'Seed': 'SeedL'})

In [ ]:
def treat_seed(seed):
    return int(re.sub("[^0-9]", "", seed))

In [ ]:
df['SeedW'] = df['SeedW'].apply(treat_seed)
df['SeedL'] = df['SeedL'].apply(treat_seed)

In [ ]:
df.head(10)


In [ ]:
df = pd.merge(
    df,
    df_features_season,
    how='left',
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID']
).rename(columns={
    'NumWins': 'NumWinsW',
    'NumLosses': 'NumLossesW',
    'GapWins': 'GapWinsW',
    'GapLosses': 'GapLossesW',
    'WinRatio': 'WinRatioW',
    'GapAvg': 'GapAvgW',
}).drop(columns='TeamID', axis=1)

In [ ]:
df = pd.merge(
    df,
    df_features_season,
    how='left',
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID']
).rename(columns={
    'NumWins': 'NumWinsL',
    'NumLosses': 'NumLossesL',
    'GapWins': 'GapWinsL',
    'GapLosses': 'GapLossesL',
    'WinRatio': 'WinRatioL',
    'GapAvg': 'GapAvgL',
}).drop(columns='TeamID', axis=1)

In [ ]:
df.head(10)


# Change Columns names W,L to A,B

In [ ]:
def add_loosing_matches(df):
    win_rename = {
        "WTeamID": "TeamIdA", 
        "WScore" : "ScoreA", 
        "LTeamID" : "TeamIdB",
        "LScore": "ScoreB",
     }
    win_rename.update({c : c[:-1] + "A" for c in df.columns if c.endswith('W')})
    win_rename.update({c : c[:-1] + "B" for c in df.columns if c.endswith('L')})
    
    lose_rename = {
        "WTeamID": "TeamIdB", 
        "WScore" : "ScoreB", 
        "LTeamID" : "TeamIdA",
        "LScore": "ScoreA",
    }
    lose_rename.update({c : c[:-1] + "B" for c in df.columns if c.endswith('W')})
    lose_rename.update({c : c[:-1] + "A" for c in df.columns if c.endswith('L')})
    
    win_df = df.copy()
    lose_df = df.copy()
    
    win_df = win_df.rename(columns=win_rename)
    lose_df = lose_df.rename(columns=lose_rename)
    
    return pd.concat([win_df, lose_df], axis=0, sort=False)

In [ ]:
df = add_loosing_matches(df)


In [ ]:
df.head(10)


# Making Features about Diff

In [ ]:
cols_to_diff = [
    'Seed', 'WinRatio', 'GapAvg', # '538rating'
]

for col in cols_to_diff:
    df[col + 'Diff'] = df[col + 'A'] - df[col + 'B']

# Aggregating Test data 
Sample Submission from March Madness 2023 ( Same as 2022 )


In [ ]:
df_test = pd.read_csv("/kaggle/input/2023-march-sub/SampleSubmission2023.csv")


In [ ]:
df_test

In [ ]:
%%time
def separate_id(df):
    """
    
    """
    df['Season']  = df['ID'].apply(lambda x: int(x.split('_')[0]))
    df['TeamIdA'] = df['ID'].apply(lambda x: int(x.split('_')[1]))
    df['TeamIdB'] = df['ID'].apply(lambda x: int(x.split('_')[2]))
    return df

df_test = separate_id(df_test)

In [ ]:
%%time

df_test = pd.merge(
    df_test,
    df_seeds,
    how='left',
    left_on=['Season', 'TeamIdA'],
    right_on=['Season', 'TeamID']
).drop('TeamID', axis=1).rename(columns={'Seed': 'SeedA'}).fillna('W01')

df_test = pd.merge(
    df_test, 
    df_seeds, 
    how='left', 
    left_on=['Season', 'TeamIdB'], 
    right_on=['Season', 'TeamID']
).drop('TeamID', axis=1).rename(columns={'Seed': 'SeedB'}).fillna('W01')
df_test['SeedA'] = df_test['SeedA'].apply(treat_seed)
df_test['SeedB'] = df_test['SeedB'].apply(treat_seed)
df_test.head(30)

In [ ]:
%%time

df_test = pd.merge(
    df_test,
    df_features_season,
    how='left',
    left_on=['Season', 'TeamIdA'],
    right_on=['Season', 'TeamID']
).rename(columns={
    'NumWins': 'NumWinsA',
    'NumLosses': 'NumLossesA',
    'GapWins': 'GapWinsA',
    'GapLosses': 'GapLossesA',
    'WinRatio': 'WinRatioA',
    'GapAvg': 'GapAvgA',
}).drop(columns='TeamID', axis=1)

df_test = pd.merge(
    df_test,
    df_features_season,
    how='left',
    left_on=['Season', 'TeamIdB'],
    right_on=['Season', 'TeamID']
).rename(columns={
    'NumWins': 'NumWinsB',
    'NumLosses': 'NumLossesB',
    'GapWins': 'GapWinsB',
    'GapLosses': 'GapLossesB',
    'WinRatio': 'WinRatioB',
    'GapAvg': 'GapAvgB',
}).drop(columns='TeamID', axis=1)


In [ ]:
cols_to_diff = [
    'Seed', 'WinRatio', 'GapAvg', # '538rating'
]
for col in cols_to_diff:
    df_test[col + 'Diff'] = df_test[col + 'A'] - df_test[col + 'B']
    
# Compute Difference in Final Score (ScoreDiff) and whether or not the team won (WinA)
df['ScoreDiff'] = df['ScoreA'] - df['ScoreB']
df['WinA'] = (df['ScoreDiff'] > 0).astype(int)

In [ ]:
df_test

# Building Model - Xgboost
previous solution from
https://www.kaggle.com/code/cv13j0/ncaa-gradient-boosted-trees-xgboost#Creating-the-Test-Dataset by @cv13j0

In [ ]:
from sklearn import tree
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import log_loss
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [ ]:
target_feature = 'WinA'
avoid = ['ScoreDiff', 'Season', 'DayNum', 'A_Win']
features = [col for col in df.columns if col not in avoid]

In [ ]:
features

In [ ]:
features = ['TeamIdA',
            #'ScoreA',
            'TeamIdB',
            #'ScoreB',
            'SeedA',
            'SeedB',
            'WinRatioA',
            'GapAvgA',
            'WinRatioB',
            'GapAvgB',
            'SeedDiff',
            'WinRatioDiff',
            'GapAvgDiff']

In [ ]:
%%time
# Develop a CV loop to avoid leaking data from future tournaments...
def kfold_model(train_df, tst_df):
    cvs = []
    preds_test = []
    seasons = train_df['Season'].unique()
    
    for season in seasons[1:]:
        print(f'\nValidating on season {season}')
        X_train = train_df[train_df['Season'] < season][features].reset_index(drop = True).copy()
        X_val = train_df[train_df['Season'] == season][features].reset_index(drop = True).copy()
        
        y_train = train_df[train_df['Season'] < season][target_feature].reset_index(drop = True).copy()
        y_val = train_df[train_df['Season'] == season][target_feature].reset_index(drop = True).copy()
        
        tst_dataset = tst_df[features].copy()
        
        
        scaler = MinMaxScaler()
        scaler.fit(X_train)
        
        X_train = scaler.transform(X_train)        
        X_val = scaler.transform(X_val)
        tst_dataset = scaler.transform(tst_dataset)
        
        model = XGBClassifier(n_estimators = 1024, random_state = 85)
        model.fit(X_train, y_train, eval_set = [(X_val, y_val)], verbose = 0, early_stopping_rounds = 128)
        pred = model.predict_proba(X_val)[:, 1]
        
        pred_test = model.predict_proba(tst_dataset)[:, 1]
        preds_test.append(pred_test)
        
        loss = log_loss(y_val, pred)
        cvs.append(loss)
        
        print(f'\t -> Scored {loss:.4f}')
    print(f'\nLocal Cross Validation Score Is: {np.mean(cvs):.3f}', '\n')
    return preds_test

In [ ]:
%%time
predictions = kfold_model(df, df_test)

In [ ]:
mean_predictions = np.mean(predictions, 0)

sub = df_test[['ID', 'Pred']].copy()
sub['Pred'] = mean_predictions

In [ ]:
sub

# Simulating predictions till round 6
so that u can change prediction format to bracket format
- How to apply previous solutions to March Madness 2024 : https://www.kaggle.com/code/samdaltonjr/preliminary-preds-into-bracket by [@samdaltonjr](https://www.kaggle.com/samdaltonjr)

In [ ]:
seeds_2024=pd.read_csv(DATA_PATH+'2024_tourney_seeds.csv')
seeds_2024['Year']=2023
seeds_2024

In [ ]:
sub['Year'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
sub['Team1ID'] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
sub['Team2ID'] = sub['ID'].apply(lambda x: int(x.split('_')[2]))


In [ ]:
# Merge seeds_2024 with sub for both teams
sub = sub.merge(seeds_2024, left_on=['Year', 'Team1ID'], right_on=['Year', 'TeamID'], how='left')
sub.rename(columns={'Seed': 'Team1Seed'}, inplace=True)
sub.drop('TeamID', axis=1, inplace=True)

sub = sub.merge(seeds_2024, left_on=['Year', 'Team2ID'], right_on=['Year', 'TeamID'], how='left')
sub.rename(columns={'Seed': 'Team2Seed', 'Tournament_x':'Tournament'}, inplace=True)
sub.drop(['TeamID', 'Tournament_y'], axis=1, inplace=True)

In [ ]:
sub=sub.dropna()
sub

# Preprocessing of Submission file

In [ ]:
#This cell is where the transformation of your original prediction file takes place, it flips the {Year}_{Team1ID}_{Team2ID} format into {Year}_{HigherSeed}_{LowerSeed}
#An Additional Column new_ID will be created to comtain the original Team IDs

preds_w_seeds = sub.copy()

#sort preds_w_seeds by Team1Seed
preds_w_seeds = preds_w_seeds.sort_values(by='Team1Seed')


#Flip preds to where pred is based on the higher seed and not lower seed, so pred must be transformed accordingly but new_ID would be formated as 2023_X01_X16, etc. only where pred is not already in the correct format

def extract_seed_number(seed):
    return int(seed[1:])

# Assuming preds_w_seeds is your DataFrame
# Define a function to compare seeds
def compare_seeds(seed1, seed2):
    seed1_num = extract_seed_number(seed1)
    seed2_num = extract_seed_number(seed2)
    seed1_prefix = seed1[0]
    seed2_prefix = seed2[0]

    if seed1_num < seed2_num:
        return -1
    elif seed1_num > seed2_num:
        return 1
    else:  # If seed numbers are equal, compare prefixes
        if seed1_prefix < seed2_prefix:
            return -1
        elif seed1_prefix > seed2_prefix:
            return 1
        else:
            return 0

# Assuming preds_w_seeds is your DataFrame
# First, identify higher seed and lower seed for each matchup
def determine_seeds(row):
    seed1 = row['Team1Seed']
    seed2 = row['Team2Seed']

    comparison = compare_seeds(seed1, seed2)
    if comparison < 0:
        row['HigherSeed'] = row['Team1ID']
        row['HigherSeedID'] = seed1
        row['LowerSeed'] = row['Team2ID']
        row['LowerSeedID'] = seed2
    elif comparison > 0:
        row['HigherSeed'] = row['Team2ID']
        row['HigherSeedID'] = seed2
        row['LowerSeed'] = row['Team1ID']
        row['LowerSeedID'] = seed1
    else:  # If seeds are equal
        if row['Team1ID'] < row['Team2ID']:
            row['HigherSeed'] = row['Team1ID']
            row['HigherSeedID'] = seed1
            row['LowerSeed'] = row['Team2ID']
            row['LowerSeedID'] = seed2
        else:
            row['HigherSeed'] = row['Team2ID']
            row['HigherSeedID'] = seed2
            row['LowerSeed'] = row['Team1ID']
            row['LowerSeedID'] = seed1

    return row

preds_w_seeds = preds_w_seeds.apply(determine_seeds, axis=1)

# Then, rearrange the data to create new ID and update Pred column accordingly
preds_w_seeds['ID'] = preds_w_seeds.apply(lambda x: f"{x['Year']}_{x['HigherSeed']}_{x['LowerSeed']}", axis=1)
preds_w_seeds['Pred'] = preds_w_seeds.apply(lambda x: 1 - x['Pred'] if x['Team1ID'] != x['HigherSeed'] else x['Pred'], axis=1)  # Flip Pred only if teams are rearranged
preds_w_seeds = preds_w_seeds[['ID', 'Pred', 'Year', 'HigherSeedID', 'LowerSeedID', 'Tournament', 'HigherSeed', 'LowerSeed']]

preds_w_seeds['new_ID'] = preds_w_seeds['Tournament'] + '_' + preds_w_seeds['Year'].astype(str) + '_' + preds_w_seeds['HigherSeedID'] + '_' + preds_w_seeds['LowerSeedID']

In [ ]:
# Function to simulate a single matchup
def simulate_matchup(probability):
    return np.random.rand() < probability  # Returns True if Team1 wins, False if Team2 wins

In [ ]:
#sample_submission as template for the output
sample_submission = pd.read_csv(DATA_PATH+'sample_submission.csv')

submission = sample_submission.copy()
submission.rename(columns={'Team':'Winner'},inplace=True)

In [ ]:
tourney_slots = pd.read_csv(DATA_PATH+'MNCAATourneySlots.csv')
#filter out tourney_slots to only include 2023
tourney_slots = tourney_slots[tourney_slots['Season'] == 2023]

tourney_slots_M = tourney_slots.copy()
tourney_slots_M['Tournament'] = 'M'

tourney_slots_W = tourney_slots.copy()
tourney_slots_W['Tournament'] = 'W'

tourney_slots = pd.concat([tourney_slots_M, tourney_slots_W])

In [ ]:
def create_round(slot):
    if slot[0] != 'R':
        return 0
    elif slot[0] == 'R':
        return int(slot[1])

In [ ]:
tourney_slots['Round'] = tourney_slots['Slot'].apply(create_round)
tourney_slots = tourney_slots[tourney_slots['Round'] != 0]

In [ ]:
preds_df = preds_w_seeds[['ID','new_ID','Year','HigherSeedID','LowerSeedID','Tournament','Pred']]

preds_df

In [ ]:
#merge preds_df with initial_matchups to get the pred for round 1 matchup

tourney_2023 = tourney_slots.copy()

#find round 1 matchups
initial_matchups = tourney_2023[tourney_2023['Round'] == 1]

results = initial_matchups.merge(preds_df, left_on=['StrongSeed','WeakSeed','Season'], right_on=['HigherSeedID','LowerSeedID','Year'], how='left')

results.rename(columns={'Tournament_x':'Tournament'}, inplace=True)
#drop Touranment_y column  
results.drop('Tournament_y', axis=1, inplace=True)

In [ ]:
results

# Simulate Round 1 

In [ ]:
#simulate round 1 matchups, if simulate_matchup results in True, winner is the higher seed, if False, winner is the lower seed
results['outcome'] = results['Pred'].apply(simulate_matchup)

results['Team'] = results.apply(lambda x: x['HigherSeedID'] if x['outcome'] == True else x['LowerSeedID'], axis=1)

#drop duplicates from results
results.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

results

In [ ]:
round_1_results = results[['Tournament','Slot','Team','ID']]

round_1_results

# Simulate Round 2

In [ ]:
#take Team from round_1_results and merge with tourney_2023 to get the next round matchups
round_2 = tourney_2023[tourney_2023['Round'] == 2]

In [ ]:
round_2 = round_2.merge(round_1_results, left_on=['StrongSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_2.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_2.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_2.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

In [ ]:
round_2 = round_2.merge(round_1_results, left_on=['WeakSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_2.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_2.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_2.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

In [ ]:
#if team_x is the lower seed, set StrongSeed to Team_x and WeakSeed to Team_y, else set StrongSeed to Team_y and WeakSeed to Team_x
round_2['StrongSeed'] = round_2.apply(lambda x: x['Team_x'] if x['Team_x'] < x['Team_y'] else x['Team_y'], axis=1)
round_2['WeakSeed'] = round_2.apply(lambda x: x['Team_x'] if x['Team_x'] > x['Team_y'] else x['Team_y'], axis=1)

#drop Team_x and Team_y
round_2.drop(['Team_x', 'Team_y'], axis=1, inplace=True)

In [ ]:
#merge round_2 with preds_df to get the pred for round 2 matchups
round_2 = round_2.merge(preds_df, left_on=['StrongSeed','WeakSeed','Tournament'], right_on=['HigherSeedID','LowerSeedID','Tournament'], how='left')

round_2['outcome'] = round_2['Pred'].apply(simulate_matchup)

round_2['Team'] = round_2.apply(lambda x: x['HigherSeedID'] if x['outcome'] == True else x['LowerSeedID'], axis=1)

round_2

In [ ]:
round_2_results = round_2[['Tournament','Slot','Team','ID']]

round_2_results

# Simulate Round 3

In [ ]:
round_3 = tourney_2023[tourney_2023['Round'] == 3]

round_3 = round_3.merge(round_2_results, left_on=['StrongSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_3.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_3.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_3.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

round_3 = round_3.merge(round_2_results, left_on=['WeakSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_3.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_3.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_3.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

In [ ]:
round_3['StrongSeed'] = round_3.apply(lambda x: x['Team_x'] if x['Team_x'] < x['Team_y'] else x['Team_y'], axis=1)
round_3['WeakSeed'] = round_3.apply(lambda x: x['Team_x'] if x['Team_x'] > x['Team_y'] else x['Team_y'], axis=1)

#drop Team_x and Team_y
round_3.drop(['Team_x', 'Team_y'], axis=1, inplace=True)

In [ ]:
round_3 = round_3.merge(preds_df, left_on=['StrongSeed','WeakSeed','Tournament'], right_on=['HigherSeedID','LowerSeedID','Tournament'], how='left')

round_3['outcome'] = round_3['Pred'].apply(simulate_matchup)

round_3['Team'] = round_3.apply(lambda x: x['HigherSeedID'] if x['outcome'] == True else x['LowerSeedID'], axis=1)

round_3

In [ ]:
round_3_results = round_3[['Tournament','Slot','Team','ID']]
round_3_results

# Simulate Round 4

In [ ]:
round_4 = tourney_2023[tourney_2023['Round'] == 4]

round_4 = round_4.merge(round_3_results, left_on=['StrongSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_4.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_4.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_4.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

round_4 = round_4.merge(round_3_results, left_on=['WeakSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_4.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_4.drop(['Slot_y'], axis=1, inplace=True)
    
#drop duplicates
round_4.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

In [ ]:
round_4['StrongSeed'] = round_4.apply(lambda x: x['Team_x'] if x['Team_x'] < x['Team_y'] else x['Team_y'], axis=1)
round_4['WeakSeed'] = round_4.apply(lambda x: x['Team_x'] if x['Team_x'] > x['Team_y'] else x['Team_y'], axis=1)

#drop Team_x and Team_y
round_4.drop(['Team_x', 'Team_y'], axis=1, inplace=True)

In [ ]:
round_4 = round_4.merge(preds_df, left_on=['StrongSeed','WeakSeed','Tournament'], right_on=['HigherSeedID','LowerSeedID','Tournament'], how='left')

round_4['outcome'] = round_4['Pred'].apply(simulate_matchup)

round_4['Team'] = round_4.apply(lambda x: x['HigherSeedID'] if x['outcome'] == True else x['LowerSeedID'], axis=1)

round_4

In [ ]:
round_4_results = round_4[['Tournament','Slot','Team','ID']]
round_4_results

# Simulate Round 5

In [ ]:
round_5 = tourney_2023[tourney_2023['Round'] == 5]

round_5 = round_5.merge(round_4_results, left_on=['StrongSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_5.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_5.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_5.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

round_5 = round_5.merge(round_4_results, left_on=['WeakSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_5.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_5.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_5.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

In [ ]:
def compare_seeds(seed1, seed2):
    seed_num1 = extract_seed_number(seed1)
    seed_num2 = extract_seed_number(seed2)

    if seed_num1 == seed_num2:
        # If the seed numbers are equal, compare the letters
        return seed1 if seed1 < seed2 else seed2
    else:
        # Otherwise, compare the seed numbers
        return seed1 if seed_num1 < seed_num2 else seed2

# Apply comparison logic to determine StrongSeed and WeakSeed
round_5['StrongSeed'] = round_5.apply(lambda x: compare_seeds(x['Team_x'], x['Team_y']), axis=1)
round_5['WeakSeed'] = round_5.apply(lambda x: x['Team_x'] if x['Team_x'] != x['StrongSeed'] else x['Team_y'], axis=1)

#drop Team_x and Team_y
round_5.drop(['Team_x', 'Team_y'], axis=1, inplace=True)
round_5

In [ ]:
round_5 = round_5.merge(preds_df, left_on=['StrongSeed','WeakSeed','Tournament'], right_on=['HigherSeedID','LowerSeedID','Tournament'], how='left')

round_5['outcome'] = round_5['Pred'].apply(simulate_matchup)

round_5['Team'] = round_5.apply(lambda x: x['HigherSeedID'] if x['outcome'] == True else x['LowerSeedID'], axis=1)

round_5

In [ ]:
round_5_results = round_5[['Tournament','Slot','Team','ID']]

round_5_results

# Simulate Round 6

In [ ]:
round_6 = tourney_2023[tourney_2023['Round'] == 6]

round_6 = round_6.merge(round_5_results, left_on=['StrongSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_6.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)
#drop columns from merge
round_6.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_6.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

round_6 = round_6.merge(round_5_results, left_on=['WeakSeed','Tournament'], right_on=['Slot','Tournament'], how='left')
#rename original columns
round_6.rename(columns={'Slot_x':'Slot', 'Tournament_x':'Tournament'}, inplace=True)    
#drop columns from merge
round_6.drop(['Slot_y'], axis=1, inplace=True)

#drop duplicates
round_6.drop_duplicates(subset=['Slot','Tournament'], inplace=True)

In [ ]:
# Apply comparison logic to determine StrongSeed and WeakSeed
round_6['StrongSeed'] = round_6.apply(lambda x: compare_seeds(x['Team_x'], x['Team_y']), axis=1)
round_6['WeakSeed'] = round_6.apply(lambda x: x['Team_x'] if x['Team_x'] != x['StrongSeed'] else x['Team_y'], axis=1)

round_6.drop(['Team_x', 'Team_y'], axis=1, inplace=True)

In [ ]:
round_6 = round_6.merge(preds_df, left_on=['StrongSeed','WeakSeed','Tournament'], right_on=['HigherSeedID','LowerSeedID','Tournament'], how='left')

round_6['outcome'] = round_6['Pred'].apply(simulate_matchup)

round_6['Team'] = round_6.apply(lambda x: x['HigherSeedID'] if x['outcome'] == True else x['LowerSeedID'], axis=1)

round_6

In [ ]:
round_6_results = round_6[['Tournament','Slot','Team','ID']]

round_6_results

# Concatenate All Round Results

In [ ]:
#concatentate rounds 1-6 results together
all_rounds = pd.concat([round_1_results, round_2_results, round_3_results, round_4_results, round_5_results, round_6_results])

all_rounds

In [ ]:
print(all_rounds)


# Final Submission

In [ ]:
#sample_submission as template for the output

df = sample_submission.copy()

#drop Team column from sample_submission
df.drop('Team', axis=1, inplace=True)

#merge sample_submission with all_rounds on Tournament and Slot
submission = df.merge(all_rounds, left_on=['Tournament', 'Slot'], right_on=['Tournament', 'Slot'], how='left')

#drop ID from submission
submission.drop('ID', axis=1, inplace=True)

submission

In [ ]:
#return all null rows from submission
submission[submission['Team'].isnull()]

In [ ]:
submission.to_csv('submission.csv', index=False)